In [ ]:
import os
import typing
from datasets import load_dataset, Dataset, IterableDataset
from functools import reduce
chime_path = "/home/niklas/Downloads/Datasets/CHIME6/CHiME6_eval/CHiME6/audio/eval"
dipco_path = "/home/niklas/Downloads/Datasets/Dipco/"

import os
from datasets import Dataset, Audio
import pandas as pd


def create_dipco_from_directory(directory_path, output_path):
    dev_path = os.path.join(dipco_path, 'audio/dev')
    transcript_dev_path = os.path.join(dipco_path, 'transcriptions/dev')
    #dev_audio_files = [os.path.join(directory_path, os.path.join(dev_path, file)) for file in os.listdir(dev_path) if file.endswith(('.wav', '.mp3', '.flac'))]
    dev_transcript_files = [os.path.join(transcript_dev_path, file) for file in os.listdir(transcript_dev_path) if file.endswith(('.json'))]
    print(dev_transcript_files[0])
    df_devs = [pd.read_json(jsonfile) for jsonfile in dev_transcript_files]
 
    final_dev = reduce(lambda left,right: pd.merge(left,right, on=['labels'], how='outer'), df_devs)
    
    #df = pd.DataFrame(dev_audio_files, columns=["file_path"])
    #dataset = Dataset.from_pandas(df)
    #dataset = dataset.cast_column("file_path", Audio(sampling_rate=16000))
    
    #dataset.save_to_disk(output_path)

directory_path = dipco_path
output_path = dipco_path
create_dipco_from_directory(directory_path, output_path)

    
"""
def create_audio_dataset_from_directory(directory_path, output_path):
    
    Create a Hugging Face dataset from a directory of audio files.

    Args:
        directory_path (str): Path to the directory containing audio files.
        output_path (str): Path to save the output dataset.
    
    # Get all audio file paths
    audio_files = [os.path.join(directory_path, file) for file in os.listdir(directory_path) if file.endswith(('.wav', '.mp3', '.flac'))]

    # Create a dataframe with file paths
    df = pd.DataFrame(audio_files, columns=["file_path"])
 
    # Convert dataframe to Hugging Face dataset
    dataset = Dataset.from_pandas(df)

    # Cast the 'file_path' column to 'audio' feature type
    dataset = dataset.cast_column("file_path", Audio(sampling_rate=16000))

    # Save the dataset
    dataset.save_to_disk(output_path)

    print(f"Dataset saved to {output_path}")

# Usage
directory_path = chime_path
output_path = os.getcwd()
create_audio_dataset_from_directory(directory_path, output_path)

"""



In [3]:
import os  
import pandas as pd 
import torchaudio 
import re 
from typing import List
dipco_path = "/home/niklas/Downloads/Datasets/Dipco/"    
dev_path = os.path.join(dipco_path, 'audio/dev')
transcript_dev_path = os.path.join(dipco_path, 'transcriptions/dev')
file_path = "S02.json"

def extract_prefix(file_path:str) -> str:
    pattern = r'^(.*)\.json$'
    match = re.search(pattern, file_path)
    if match:
        prefix = match.group(1)
        return prefix
    else :
        raise ValueError
    
session = extract_prefix(file_path=file_path)
print(session)  



full_path = os.path.join(transcript_dev_path, file_path)
df = pd.read_json(full_path)
transcriptions = df['words']

print(df.columns)
print(df['start_time'].head(1))
print(full_path)



ModuleNotFoundError: No module named 'pandas'

In [2]:
from transformers import WhisperFeatureExtractor
from typing import Dict
import torch 
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
def expand_start_time(row):
    start_time_dict = row['start_time']
    rows = []
    for key, time_str in start_time_dict.items():
        new_row = row.copy()
        new_row['audio'] = key
        new_row['start'] = time_str
        rows.append(new_row)
    return pd.DataFrame(rows)


# Apply the function to each row and concatenate the results
expanded_df = pd.concat([expand_start_time(row) for _, row in df.iterrows()], ignore_index=True)

# Drop the original 'start_time' column
expanded_df = expanded_df.drop(columns=['start_time'])

# Function to convert time string to seconds
def time_to_seconds(time_str):
    h, m, s = map(float, time_str.split(':'))
    
    return h * 3600 + m * 60 + s

# Apply the conversion to the 'start' column
expanded_df['start'] = expanded_df['start'].apply(time_to_seconds)

def get_corresponding_end_time(dict:dict, key:str):
    end_time = [v for k,v in dict if k==key]
    return end_time
print(expanded_df.columns)
expanded_df['end'] = expanded_df.apply(lambda row: row['end_time'][row['audio']], axis=1)
expanded_df['end'] = expanded_df['end'].apply(time_to_seconds)
# removal of the end_time
expanded_df = expanded_df.drop(columns=['end_time'])
#expanded_df = expanded_df.drop(expanded_df['audio']=='close-talk')


# U01 - U05 & CH 1 - 7 

# Function to generate microphone paths
def generate_microphone_paths(row):
    paths = []
    for i in range(1, 7):
        path = f"{dev_path}/{row['session_id']}_{row['audio']}.CH{i}.wav"
        paths.append(path)

    path = f"{dev_path}/{row['session_id']}_{row['speaker_id']}.wav"
    paths.append(path)
    return paths

# Apply the function to generate the paths for each row
expanded_df['file_path'] = expanded_df.apply(generate_microphone_paths, axis=1)

# Expand the DataFrame to include the microphone paths
expanded_df = expanded_df.explode('file_path').reset_index(drop=True)


#change the seconds to frames
def get_Frames(starting_second:float, sample_rate:int, end_second:float )-> List[int] :
     return [int(starting_second*sample_rate), int(end_second)*sample_rate]

expanded_df['frames'] = expanded_df.apply(lambda row: get_Frames(row['start'], 16000, row['end']), axis=1)
expanded_df = expanded_df[expanded_df['audio'] != 'close-talk']
columns_to_drop = ['mother_tongue', 'ref', 'nativeness', 'audio', 'session_id','speaker_id', 'gender',  'end']
expanded_df = expanded_df.drop(columns=columns_to_drop)
# sorting for cache efficiency so far no speedup 
def validate_frames_column(frames_list):
    return len(frames_list) == 2
if expanded_df['frames'].isnull().any():
    raise ValueError("The 'frames' column contains null values.")
if not expanded_df['frames'].apply(validate_frames_column).all():
    raise ValueError("Each entry in the 'frames' column must be a list of exactly two elements [startframe, endframe].")

expanded_df[['startframe', 'endframe']] = pd.DataFrame(expanded_df['frames'].tolist(), index=expanded_df.index)

# Drop the original 'frames' column if no longer needed
"""expanded_df = expanded_df.drop(columns=['frames'])

expanded_df = expanded_df.sort_values(by=['file_path','start'])
expanded_df = expanded_df.reset_index(drop=True)
grouped = expanded_df.groupby(['words'])
count_df = grouped.size().reset_index(name='counts')
first_group_key = list(grouped.groups.keys())[0]
first_group = grouped.get_group(first_group_key)
print(first_group)
print(first_group['file_path'].value_counts())
print(count_df)"""

print(expanded_df.shape)
expanded_df = expanded_df.head(15)
print(expanded_df.columns)
expanded_df['logmel'] = expanded_df.apply(lambda row: get_logmel(row['startframe'], row['endframe'], row['file_path']), axis=1)
def get_logmel(startframe: int, endframe: int, filepath: str) -> Dict[str, torch.Tensor]:
    sliced_waveform = load_audio_segment(filepath=filepath, start_frame=startframe, end_frame=endframe)
    features = feature_extractor(sliced_waveform.numpy(), sampling_rate=16000, return_tensors='pt')
    return features


    
def load_audio_segment(filepath, start_frame, end_frame):
    waveform, sample_rate = torchaudio.load(filepath)
    return waveform[:, start_frame:end_frame], sample_rate    
#print(expanded_df)
#print(expanded_df.head(10))



Index(['end_time', 'words', 'ref', 'session_id', 'speaker_id', 'gender',
       'nativeness', 'mother_tongue', 'audio', 'start'],
      dtype='object')
(15680, 6)
Index(['words', 'start', 'file_path', 'frames', 'startframe', 'endframe'], dtype='object')


NameError: name 'get_logmel' is not defined

In [ ]:
import torch
import cProfile
from transformers import WhisperFeatureExtractor
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")


dataset = Dataset.from_pandas(expanded_df.head(15))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Cast the 'file_path' column to 'audio' feature type
dataset = dataset.cast_column("file_path", Audio(sampling_rate=16000))
#print(dataset.features)
def load_audio_segment(filepath, start_frame, end_frame):
    waveform, sample_rate = torchaudio.load(filepath)
    return waveform[:, start_frame:end_frame], sample_rate
def profile_audio_loading(df):
    load_audio_segment(df['file_path'][0], df['startframe'][0], df['endframe'][0])
# slicing operation is fast 
cProfile.run("load_audio_segment(filepath=expanded_df['file_path'][0], start_frame=expanded_df['startframe'][0], end_frame=expanded_df['endframe'][0])")
def extract_audio(batch):
    
    feature_list = []
    sample_rate_list = []
    
    for idx in range(len(batch['file_path'])):
        filepath = batch['file_path'][idx]['path']
        start_frame = batch['startframe'][idx]
        end_frame = batch['endframe'][idx]
        
        # Ensure filepath is a string
        if isinstance(filepath, dict):
            print(file_path)
        
        # Load audio segment from file path
        waveform, sample_rate = load_audio_segment(filepath, start_frame, end_frame)
        features = feature_extractor(waveform.numpy(), sampling_rate=16000, return_tensors='pt')

        
        # Append loaded waveform and sample rate to lists
        feature_list.append(features)
        sample_rate_list.append(sample_rate)
    
    # Assign audio and sample rate lists to batch
    batch['logmel'] = feature_list
    batch['sample_rate'] = sample_rate_list
    
    return batch
#cProfile.run("dataset.map(extract_audio, batched=True, batch_size=1, num_proc=1, load_from_cache_file=True)")

dataset = dataset.map(extract_audio, batched=True, batch_size=1, num_proc=1, load_from_cache_file=True)

# Save the processed dataset to the downloads folder
dataset.save_to_disk('./Downloads/processed_dataset')